# PREPROCESSING STEP FOR THE BMI

Analysis of the calcium activity during the baseline period (typically 10-15min). Steps:
1) Compute the std/correlation map
2) Segment the cells: the segmentation step can be done with three different approaches
    - Otsu thresholding and watershed
    - StarDist plugin
    - Manual drawing of the ROIs
3) Compute the traces of the segmented cells
4) Choose 2 neurons per ensemble
5) Calculate minimum and maximum thresholds of activity
6) Save all the parameters


Analyse the baseline data to detect candidate cells and select cells for the ensembles.

## Library import

In [1]:
## Library import

import matplotlib
matplotlib.use('qt5agg')
%matplotlib qt5

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload

%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils.utils import ComputeROIs


import numpy as np
from skimage import data
from skimage.filters import threshold_otsu, gaussian
from skimage.segmentation import clear_border
from skimage.measure import label, regionprops_table
from skimage.morphology import closing, square, remove_small_objects

import napari
import cv2


from napari_tools_menu import register_function
from napari_time_slicer import time_slicer
from magicgui import magicgui

from matplotlib.backends.backend_qt5agg import FigureCanvas
from matplotlib.figure import Figure

Autosaving every 180 seconds


Cannot move to target thread (0x557a2623cdb0)


Available platform plugins are: xcb, eglfs, linuxfb, minimal, minimalegl, offscreen, vnc, wayland-egl, wayland, wayland-xcomposite-egl, wayland-xcomposite-glx, webgl.



## Load the BMI data

In [5]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
#fname = r"H:\Downloads\image_1000frames.raw"
#fname = '/home/cat/data/donato/bscope_tests/image_10000frames.raw'
#fname = '/media/cat/4TBSSD/donato/Bscope_tests/image_10000frames.raw'
#fname = '/media/cat/4TB/donato/BSCOPE_tests/image_27000frames.raw'

#fname = r'Z:\RG Donato\Microscopy\Catalin\DON-008498\mouse1_calibration\Image_001_001.raw'
fname = r'C:\Users\Mariona\Documents\data\mouse1_calibration\Image_001_001.raw'
gr = ComputeROIs(fname)

#
gr.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


In [6]:

data = np.memmap(fname, dtype='uint16', mode='r')
data = data.reshape(-1,512,512)
print ("memmap : ", data.shape)

data_sparse = data[::gr.subsample]
print ("data into analysis: ", data_sparse.shape)

sigma = 1
order = 0
print (" gaussian filter width: ", sigma, ", order: ", order)
data_sparse = scipy.ndimage.gaussian_filter(data_sparse, 
                                            sigma, 
                                            order)

memmap :  (20000, 512, 512)
data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0


## Compute the correlation map

Still very slow. Not need to run it

In [ ]:
def compute_corr_neighbours(data, row,col, indexes, n_neighbors=2):
    corr = 0
    for index in indexes:
        corr = corr+np.corrcoef(data[:,row, col], data[:,row+index[0],col+index[1]])[1,0]
    return corr


corr_map = np.ones((512,512))
n_neighbors = 1
x = np.arange(-n_neighbors, n_neighbors+1)
indexes = np.array(np.meshgrid(x,x)).T.reshape(-1,2)
data_sparse_pad = np.pad(data_sparse,n_neighbors, 'mean')

for index, elem in np.ndenumerate(data_sparse[0]):
    corr_map[index[0],index[1]] = compute_corr_neighbours(data_sparse_pad, index[0]+n_neighbors,index[1]+n_neighbors, indexes, n_neighbors=n_neighbors)

    
# Show correlation map    
plt.figure()
plt.imshow(corr_map)

In [7]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time
#      - try to speed up
#      - also, corelation map might be even better method for detecting ROIs
#         e.g. see suite2p's visualization tool <- seems pretty quick

# Also, need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()
std_map = gr.make_std_map()
print ("total processing time: ", time.time()-start, " sec")

memmap :  (20000, 512, 512)
data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
staring computing std...
done computing std...
total processing time:  18.76167917251587  sec


In [8]:
plt.close('all')

In [9]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################
gr.find_roi_boundaries(std_map)

#
gr.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
gr.trace_subsample = 10# Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()

memmap :  (20000, 512, 512)


100%|████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1077.38it/s]


In [10]:
std = gr.std_map

## Using Napari to segment the neurons and choose the best ensembles

In [11]:

@magicgui(call_button="execute")
@time_slicer
def segmentation(image:napari.types.ImageData, spot_sigma: float = 2, threshold_parameter: float = 0.1, watershed_par: int = 15) -> napari.types.ImageData:
    """
    Function to perfom image segmentation of the cells.
    Parameters:
    - Spot_sigma: value of the sigma to smooth the image
    - Threshold_parameter: the threshold can be tuned to binarise the image at different intensity values
    - Watershed_parameter: parameter to tune the watershed step
    
    Input:
    - Gray-scale image to segment
    Output:
    - Segmented image
    """
    smooth = gaussian(image, sigma = spot_sigma)
        
    thresh = threshold_otsu(smooth)
    bw = closing(smooth > thresh+threshold_parameter, square(4))
    
    
    distance = ndi.distance_transform_edt(bw)
    coords = peak_local_max(distance, 
                            footprint=np.ones((watershed_par, watershed_par)), 
                            labels=bw)

    # 
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    labels = watershed(-distance, 
                       markers, 
                       mask=bw)
    #
    labels = labels.astype('float32')

    #cleared = remove_small_objects(clear_border(labels), 20)
    
    return labels

segmentation.called.connect


<bound method SignalInstance.connect of <SignalInstance 'called' on <FunctionGui segmentation(image: <function NewType.<locals>.new_type at 0x00000209ED006F70> = None, spot_sigma: float = 2.0, threshold_parameter: float = 0.1, watershed_par: int = 15, *, viewer: napari.viewer.Viewer = None) -> <function NewType.<locals>.new_type at 0x00000209ED006F70>>>>

## Run Napari to segment cells

In [12]:
viewer = napari.view_image(std)
viewer.window.add_dock_widget(segmentation)
labels = segmentation()
#viewer = viewer.add_labels(label)

viewer.add_shapes(name='manual_annotation', face_color='yellow', edge_color='blue', opacity=0.7)

napari.run()

C:\Users\Mariona\anaconda3\lib\site-packages\napari_tools_menu\__init__.py:194: FutureWarning: Public access to Window.qt_viewer is deprecated and will be removed in
v0.5.0. It is considered an "implementation detail" of the napari
application, not part of the napari viewer model. If your use case
requires access to qt_viewer, please open an issue to discuss.
  self.tools_menu = ToolsMenu(self, self.qt_viewer.viewer)


### Choose the method to segment the cells

In [13]:
methods = ["Watershed", "StarDist", "Manual"]
method = methods[2]

In [14]:
if method == "Watershed":
    labels = segmentation()
elif method == "StarDist":
    labels = viewer.layers['StarDist labels'].data
else:
    shape = viewer.layers['std'].data.shape
    labels = viewer.layers['manual_annotation'].to_labels(labels_shape=shape)


In [15]:
min_size = 1
roi_centres = []
indexes = []
for k in np.unique(labels):
    idx = np.where(labels==k)

    if idx[0].shape[0]<50 or idx[0].shape[0]>1000:
        labels[idx]=0

    else:
        

        roi_centres.append([np.median(idx[0]),
                             np.median(idx[1])])
        indexes.append(idx)


rois = np.vstack(roi_centres)
indexes = indexes

gr.rois = np.vstack(roi_centres)
gr.indexes = indexes

In [16]:
gr.compute_traces2()

memmap :  (20000, 512, 512)


100%|████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 2490.43it/s]


## Select cells to be used for ensembles

In [17]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################



# save ensemble rois
gr.ensemble1 = [0,1]
gr.ensemble2 = [16,8]
both = np.hstack((gr.ensemble1, gr.ensemble2))


#
gr.show_traces_ids(both)


In [ ]:
## 

## Compute the min and the max for the selected ensembles


In [18]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
# # NOT SURE THIS IS NECESSARY... TO MODIFY so we can work withs subsampled series al
gr.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()

memmap :  (20000, 512, 512)


100%|██████████████████████████████████████████████████████████████████████████| 20000/20000 [00:08<00:00, 2456.65it/s]


In [19]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# 
gr.sample_rate = 30
gr.post_reward_lockout = 10   # reward lockout in seconds
gr.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
gr.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
gr.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
gr.find_reward_thresholds()
print ("thresholds: ", gr.low, gr.high)

########################################
gr.plot_rewarded_ensembles()


100%|████████████████████████████████████████████████████████████████████████| 19990/19990 [00:00<00:00, 145533.93it/s]


low, high:  -4342.794246477435 6110.496517224226
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22
updated rwards #:  22 -1087.1928725731639 1529.7266884789403
thresholds:  -1087.1928725731639 1529.7266884789403


## Save the results

In [20]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
gr.low_freq = 1000
gr.high_freq = 18000

# save cell pixel locations as 2 column array inside list
cells = []
for k in range(4):
    temp = gr.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))

    cells.append(temp)

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(fname)[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0 = cells[0],
            cell1 = cells[1],
            cell2 = cells[2],
            cell3 = cells[3],
            cell_f0s = gr.roi_f0s,
            cell_centres = np.int32(gr.rois)[both],
            cell_ids = both,
            all_rois = np.int32(gr.rois),
            low_threshold = gr.low,
            high_threshold = gr.high,
            low_freq = gr.low_freq,
            high_freq = gr.high_freq,
            cell_traces = gr.roi_traces,
            sample_rate = gr.sample_rate,
            post_reward_lockout = gr.post_reward_lockout,
            balance_ensemble_rewards_flag = gr.balance_ensemble_rewards_flag,
            rois_smooth_window = gr.rois_smooth_window,
            smooth_diff_function_flag = gr.smooth_diff_function_flag
         
        )
